# ODM Assignment 2

After coding the ODM for the first assignment 1, this assignment implements the usage of it for MongoDB queries.

## Globals

The following variables can be changed or set to modify the program's behavior:

### Data Globals

- `import_data`: If set to `True`, imports data from `json` files
- `legible_data`: If set to `True`, formats data to make it more legible in the jupyter output
- `definitions_path`: `YML` filepath for necessary models
- `scope`: Global variable that stores model classes

### MongoDB Globals

- `db_name`: MongoDB database name
- `mongodb_uri`: MongoDB URI path

It is recommended to set `import_data` to `True` if one has never ran the program before.

In [1]:
import json
import ODM
import time

from pymongo import MongoClient
from ODM import initApp
from tqdm.notebook import tqdm

# Data Globals
import_data = False
legible_data = True
definitions_path = "data/models.yml"
scope = {}

# MongoDB Globals
db_name = "abd"
mongodb_uri="mongodb://localhost:27017/"

# Initialize DB
initApp(definitions_path=definitions_path, mongodb_uri=mongodb_uri, db_name=db_name, scope=scope)

# Get model classes
User = scope.get("User")
Company = scope.get("Company")
Educational_Center = scope.get("Educational_Center")

# Introduce data if user sets introduce data variable to true
if import_data:
    print("\n=== Importing data into MongoDB ===")

    # Load JSON data
    with open("data/users.json", "r", encoding="utf-8") as f:
        users_data = json.load(f)
    with open("data/companies.json", "r", encoding="utf-8") as f:
        companies_data = json.load(f)
    with open("data/educational_centers.json", "r", encoding="utf-8") as f:
        ed_centers_data = json.load(f)

    # Helper to filter admissible/required vars
    def valid_args(model, d):
        valid = model._required_vars.union(model._admissible_vars)
        return {k: v for k, v in d.items() if k in valid}

    # User data
    users = []
    for u in tqdm(users_data, desc = "Adding users"):
        args = valid_args(User, u)
        user = User(**args)
        user.save()
        users.append(user)
        time.sleep(0.01)

    # Company data
    companies = []
    for c in tqdm(companies_data, desc = "Adding companies"):
        args = valid_args(Company, c)
        company = Company(**args)
        company.save()
        companies.append(company)
        time.sleep(0.01)

    # Educational Center data
    ed_centers = []
    for ec in tqdm(ed_centers_data, desc = "Adding educational centers"):
        args = valid_args(Educational_Center, ec)
        ed_center = Educational_Center(**args)
        ed_center.save()
        ed_centers.append(ed_center)
        time.sleep(0.01)

Connected to database: abd
Successfully connected to MongoDB!
Loading schema from data/models.yml
Initializing model: User
Initializing model: Company
Initializing model: Educational_Center


## Query 1

**List of all people who have studied at the UPM or UAM.**

- Join the `users` and the `ed_center` on the `educational_centers.email`.
- Separate the `ed_details` into documents.
- Select documents where `ed_center` name is UPM or UAM.
- Group by user `_id` to avoid duplication.
- Replace document root to only have user part.
- Project to only select what we want.

In [2]:
query1 = User.aggregate([
  {
    "$lookup": {
      "from": "Educational_Center",
      "localField": "educational_centers.email",
      "foreignField": "email",
      "as": "ed_details",
    },
  },
  { "$unwind": "$ed_details" },
  {
    "$match": {
      "ed_details.name": {
        "$in": [
          "Universidad Politécnica de Madrid",
          "Universidad Autónoma de Madrid",
        ],
      },
    },
  },
  { "$group": { "_id": "$_id", "user": { "$first": "$$ROOT"  }}},
  { "$replaceRoot": { "newRoot": "$user" }},
  {
    "$project": {
      "ed_details": 0,
    }
  }
]);

if legible_data:
    print(f"Query 1: List of all people who have studied at the UPM or UAM")
for i, doc in enumerate(query1, start=1):
    if legible_data:
        print(f"{i:2d} - {doc['name']}")
    else:
        print(doc)

Query 1: List of all people who have studied at the UPM or UAM
 1 - María Fernández
 2 - Javier López
 3 - Pedro Sánchez
 4 - Gabriel Sánchez
 5 - Esteban Castillo
 6 - Laura Martínez
 7 - Daniel Fernández
 8 - Sofía Castillo
 9 - Clara Sánchez
10 - Carlos García
11 - Juan Castillo
12 - Lucía Martínez


## Query 2

**Different universities where people residing in Madrid have studied.**

- Join with `User` using each center's email.
- Unwind joined `users` into separate docs.
- Filter `users` who live in Madrid.
- Group by educational center name and return distinct names.
- Replace document root to only have `educational_center` part.
- Project to only select what we want.

In [3]:
query2 = Educational_Center.aggregate([
  {
    "$lookup": {
      "from": "User",
      "localField": "email",                
      "foreignField": "educational_centers.email", 
      "as": "users",
    },
  },
  { "$unwind": "$users" },
  {
    "$match": {
      "users.address": { "$regex": "Madrid", "$options": "i" }, 
    },
  },
  {
    "$group": {
      "_id": "$_id",                       
      "educational_center": { "$first": "$$ROOT" }   
    },
  },
  { "$replaceRoot": { "newRoot": "$educational_center" }},
  {
    "$project": {
        "users": 0,
    }
  }
]);

if legible_data:
    print(f"Query 2: Different universities where people residing in Madrid have studied")
for i, doc in enumerate(query2, start=1):
    if legible_data:
        print(f"{i:2d} - {doc['name']}")
    else:
        print(doc)

Query 2: Different universities where people residing in Madrid have studied
 1 - Universidad Autónoma de Madrid
 2 - Universidad Complutense de Madrid
 3 - Instituto de Formación Professional López
 4 - Universidad Politécnica de Madrid


# Query 3

**People whose profile description includes "Big Data" or "Artificial Intelligence".**

- Use `$match` with `$or` to find descriptions containing the terms.
- Use case-insensitive regex to catch variations of casing.
- Replace document root to only have user part.

In [4]:
query3 = User.aggregate([
  {
    "$match": {
      "$or": [
        { "description": { "$regex": "Big Data", "$options": "i" } },
        { "description": { "$regex": "Artificial Intelligence", "$options": "i" } },
      ],
    },
  }, { "$group": { "_id": "$_id", "user": { "$first": "$$ROOT" } }},
  { "$replaceRoot": { "newRoot": "$user" }},
]);

if legible_data:
    print(f"Query 3: People whose profile description includes 'Big Data' or 'Artificial Intelligence'")
for i, doc in enumerate(query3, start=1):
    if legible_data:
        print(f"{i:2d} - {doc['name']}")
    else:
        print(doc)

Query 3: People whose profile description includes 'Big Data' or 'Artificial Intelligence'
 1 - Diego Navarro
 2 - María Fernández
 3 - Carlos García
 4 - Patricia Ruiz
 5 - Andrés Gómez
 6 - Sofía Castillo
 7 - Pedro Sánchez
 8 - Lucía Martínez


## Query 4

**Save users who completed any study in 2017 or later into a new collection.**

- Match users where at least one `educational_centers` entry has a `completion_date` $>= 2017$.
- Output results to a new collection using $out.

*NOTE:* `legible_data` *makes it so that the data is actually printed, otherwise it isn't* 

In [5]:
# Collection for output
query4col = "users_completed_studies_in_2017_or_after"
User.aggregate([
  {
    "$match": {
      "educational_centers.completion_date": { "$gte": "2017" },
    },
  },
  { "$out": query4col },
]);

if legible_data:
    # Get collection from client since we don't have YML definition of it
    client = MongoClient(mongodb_uri)
    db = client[db_name]
    col = db[query4col]

    print(f"Query 4: Save users who completed any study in 2017 or later into a new collection")
    for i, doc in enumerate(col.find(), start=1):
        print(f"{i:2d} - {doc['name']}")

Query 4: Save users who completed any study in 2017 or later into a new collection
 1 - María Fernández
 2 - Carlos García
 3 - Lucía Martínez
 4 - Javier López
 5 - Pedro Sánchez
 6 - Elena Torres
 7 - Diego Navarro
 8 - Laura González
 9 - Patricia Ruiz
10 - David Sánchez
11 - Cristina Herrera
12 - Beatriz Pérez
13 - Rocío Herrera
14 - Marta Mendoza
15 - Juan Castillo
16 - Raúl Martínez
17 - Patricia López
18 - Clara Sánchez
19 - Carla Navarro
20 - Pablo González
21 - Manuel Ruiz
22 - Alfonso Herrera
23 - Natalia Ortega
24 - Jorge Morales
25 - Claudia Jiménez
26 - Antonio Pérez
27 - Gonzalo Mendoza
28 - Sofía Castillo
29 - Emma Martínez
30 - Miguel López
31 - Gabriel Sánchez
32 - Victoria González
33 - Oscar Romero
34 - Paula Ruiz
35 - Miriam Herrera
36 - Adrián Ortega
37 - Héctor Gómez
38 - Clara Herrera
39 - Elisa Mendoza
40 - Esteban Castillo
41 - Andrea García
42 - César Martínez
43 - Irene López
44 - Aitana Sánchez
45 - Lucía González
46 - Marina Ruiz
47 - Lorena Herrera
48 - Ra

## Query 5

**Average number of studies for people who have worked at Microsoft.**

- Join users with `Company` collection via companies (company emails).
- Unwind company details and match company name to Microsoft.
- Group and compute the average number of `educational_centers` per `user`.

In [6]:
query5 = User.aggregate([
  {
    "$lookup": {
      "from": "Company",
      "localField": "companies",
      "foreignField": "email",
      "as": "c_details",
    },
  },
  { "$unwind": "$c_details" },
  { "$match": { "c_details.name": { "$regex": "Microsoft", "$options": "i" } } },
  {
    "$group": {
      "_id": "microsoft_workers",
      "average_studies": { "$avg": { "$size": "$educational_centers" } },
    },
  },
]);

if legible_data:
    print(f"Query 5: Average number of studies for people who have worked at Microsoft")
for doc in query5:
    if legible_data:
        print(f"{doc['average_studies']}")
    else:
        print(doc)

Query 5: Average number of studies for people who have worked at Microsoft
2.0


## Query 6

**Average geodesic distance to work for current Google workers.**

- Use `$geoNear` as the first stage to compute distance from a Google office coordinate (replace with the exact office coords).
- Join with `Company` collection, unwind and match company name to Google.
- Group and compute the average distance (distance stored in meters).

Replace coordinates with the correct Google office location if needed.

*NOTE: There's a difference between the query 6 provided on Friday and in this assignment due to Google coordinates being in New York (for the one on Friday) and Madrid for this assignment*

In [7]:
query6 = User.aggregate([
  {
    "$geoNear": {
      "near": { "type": "Point", "coordinates": [-3.6921, 40.4260] },
      "distanceField": "distance_from_google",
      "spherical": "true",
    },
  },
  {
    "$lookup": {
      "from": "Company",
      "localField": "companies",
      "foreignField": "email",
      "as": "c_details",
    },
  },
  { "$unwind": "$c_details" },
  { "$match": { "c_details.name": { "$regex": "Google", "$options": "i" } } },
  {
    "$group": {
      "_id": "google_workers",
      "average_distance_meters": { "$avg": "$distance_from_google" },
    },
  },
]);

if legible_data:
    print(f"Query 6: Average distance to work for current Google workers")
for doc in query6:
    if legible_data:
        print(f"{(doc['average_distance_meters'] / 1000):.3f}km")
    else:
        print(doc)

Query 6: Average distance to work for current Google workers
8.397km


## Query 7

**Top 3 universities that most often appear as study centers.**

- Join `Educational_Center` with `User` on `educational_centers.email`.
- Unwind `users` and group by `_id` counting occurrences.
- Sort by count descending and limit to the top three.
- Project to remove unwanted data.

In [8]:
query7 = Educational_Center.aggregate([
  {
    "$lookup": {
      "from": "User",
      "localField": "email",                      
      "foreignField": "educational_centers.email", 
      "as": "users",
    },
  },
  { "$unwind": "$users" },
  {
    "$group": {
      "_id": "$_id",
      "educational_center": { "$first": "$$ROOT" },
      "nb_users": { "$sum": 1 }, 
    },
  },
  { "$sort": { "nb_users": -1 } }, 
  { "$limit": 3 },
  { "$project" : { "_id": 0, "educational_center.users": 0 }}
]);

if legible_data:
    print(f"Query 7: Top 3 universities that most often appear as study centers")
for i, doc in enumerate(query7, start=1):
    if legible_data:
        print(f"{i} - {doc['educational_center']['name']} appears {doc['nb_users']} times")
    else:
        print(doc)

Query 7: Top 3 universities that most often appear as study centers
1 - Universidad Complutense de Madrid appears 10 times
2 - Centro de Estudios Médicos Valencia appears 8 times
3 - Escuela Técnica Superior de Ingeniería de Barcelona appears 8 times
